# E-INV — CONSOLIDATE
**Standalone. CPU only. No GPU, no model downloads, no regeneration.**

Reads the three completed experiment roots on Drive and recomputes **every headline number from the
raw per-row CSVs**, not from cached JSON summaries — so it is self-verifying: if a cached result and
a recomputed one disagree, this notebook tells you.

| Root | Produced by |
|---|---|
| `inv_channel/E_INV_P0_v3` | base run (S0–S7, verdict C0) |
| `inv_channel/E_SEEDEXT` | seed extension (n = 6 adapters/arm) |
| `inv_channel/E_AMP` | amplitude sweep + controls + E-POST calibration |

**Output:** an inventory of what exists, every paper number with its provenance, the paper's tables,
four figures, a consistency audit, and an explicit list of what is still missing.

Run all. A few minutes.

In [ ]:
# %% C0 — mount, locate roots, inventory
import os, json, glob, math, itertools
import numpy as np, pandas as pd
from scipy import stats as sps

try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE = "/content/drive/MyDrive"
except Exception:
    DRIVE = os.environ.get("EINV_DRIVE", ".")          # also runs off a local copy
ROOTS = {"base":    os.path.join(DRIVE, "inv_channel/E_INV_P0_v3"),
         "seedext": os.path.join(DRIVE, "inv_channel/E_SEEDEXT"),
         "amp":     os.path.join(DRIVE, "inv_channel/E_AMP"),
         "shift":   os.path.join(DRIVE, "inv_channel/E_SHIFT_REP")}   # may not exist yet

def jload(p):
    try: return json.load(open(p))
    except Exception: return None
def cload(p):
    try: return pd.read_csv(p)
    except Exception: return None

WANT = {
 "base":    ["csv/s5_measure.csv","csv/s4_copy.csv","csv/s1_positive_control.csv",
             "csv/s1b_vae_roundtrip.csv","manifest/manifest.csv",
             "fingerprints/s1_gates.json","fingerprints/s1b_vae.json",
             "fingerprints/s1c_suppression.json","E_INV_P0_results.json"],
 "seedext": ["csv/b3_measure.csv","seedext_pooled.json"],
 "amp":     ["csv/a4_measure.csv","csv/a1_real_dose.csv","csv/epost_low.csv",
             "csv/epost_lowq.csv","E_AMP_results.json"],
 "shift":   ["csv/r3_measure.csv","E_SHIFT_REP_results.json"],
}
INV = {}
print(f"{'root':9s} {'file':38s} {'status':>10s}")
for k, root in ROOTS.items():
    for f in WANT[k]:
        p = os.path.join(root, f); ok = os.path.exists(p)
        INV[(k, f)] = p if ok else None
        sz = f"{os.path.getsize(p)/1e6:.1f} MB" if ok else "MISSING"
        print(f"{k:9s} {f:38s} {sz:>10s}")
ADAPT = {k: sorted(os.path.basename(d) for d in glob.glob(os.path.join(ROOTS[k], "adapters", "*"))
                   if os.path.isdir(d) and not d.endswith(".pt")) for k in ROOTS}
for k, v in ADAPT.items():
    if v: print(f"\n{k}: {len(v)} adapters — {v}")

In [ ]:
# %% C1 — denominators, recomputed from raw CSVs
#     R_real  : device-specific PAIRED real-image contrast (NOT rho(Y_A,K_A))
#     R_VAE   : the same statistic after autoencoder round-trip
S1  = cload(INV[("base","csv/s1_positive_control.csv")])
VAE = cload(INV[("base","csv/s1b_vae_roundtrip.csv")])
NUM = {}

def real_contrast(role, other):
    a = S1[(S1.role_img==role)&(S1.role_K==role)].sort_values("idx")["rho_mult"].to_numpy(float)
    b = S1[(S1.role_img==role)&(S1.role_K==other)].sort_values("idx")["rho_mult"].to_numpy(float)
    n = min(len(a), len(b)); return a[:n]-b[:n]

RA, RB = real_contrast("A","B"), real_contrast("B","A")
allR = np.r_[RA, RB]
NUM["R_real"]    = float(0.5*(RA.mean()+RB.mean()))
NUM["R_real_se"] = float(np.sqrt((RA.var(ddof=1)/len(RA) + RB.var(ddof=1)/len(RB))/4))
NUM["R_real_L99"]= NUM["R_real"] - sps.norm.ppf(0.99)*NUM["R_real_se"]   # LOWER limit (conservative)
print(f"R_real   = {NUM['R_real']:.5e}  SE {NUM['R_real_se']:.3e}  lower99 {NUM['R_real_L99']:.5e}"
      f"   (n={len(RA)}+{len(RB)})")

if VAE is not None:
    def vae_contrast(role, stage):
        g = VAE[(VAE.role==role)&(VAE.stage==stage)].sort_values("idx")
        own, oth = ("rho_A","rho_B") if role=="A" else ("rho_B","rho_A")
        return g[own].to_numpy(float)-g[oth].to_numpy(float)
    pre  = np.r_[vae_contrast("A","pre"),  vae_contrast("B","pre")]
    post = np.r_[vae_contrast("A","post"), vae_contrast("B","post")]
    NUM["R_vae_pre"]  = float(pre.mean()); NUM["R_vae_post"] = float(post.mean())
    NUM["R_vae_se"]   = float(post.std(ddof=1)/np.sqrt(len(post)))
    NUM["R_vae_L99"]  = NUM["R_vae_post"] - sps.norm.ppf(0.99)*NUM["R_vae_se"]
    NUM["eta_device"] = NUM["R_vae_post"]/NUM["R_vae_pre"]
    print(f"R_VAE    pre {NUM['R_vae_pre']:.5e}  post {NUM['R_vae_post']:.5e} "
          f"lower99 {NUM['R_vae_L99']:.5e}")
    print(f"eta_device (device-specific VAE retention) = {NUM['eta_device']:.4f}")
g1 = jload(INV[("base","fingerprints/s1_gates.json")]) or {}
print(f"\nfor reference, the NAIVE denominator rho(Y_A,K_A) = {g1.get('rho_real_A')}"
      f"  -> R_real/naive = {NUM['R_real']/g1.get('rho_real_A',np.nan):.3f}")
print("kappa_model =", g1.get("ncc_AB"), " | same-model AUC =", g1.get("auc_pair"),
      " | pooled AUC =", g1.get("auc"))

In [ ]:
# %% C2 — the bound, recomputed at the ADAPTER level over all seeds
BASE = cload(INV[("base","csv/s5_measure.csv")])
EXT  = cload(INV[("seedext","csv/b3_measure.csv")])
ALPHA_SIM = 0.01                                   # 99% SIMULTANEOUS over the two arms

def theta_of(df, tag, ks, ko):
    a = df[(df.tag==tag)&(df.K==ks)].sort_values("gen_idx")["rho_mult"].to_numpy(float)
    b = df[(df.tag==tag)&(df.K==ko)].sort_values("gen_idx")["rho_mult"].to_numpy(float)
    n = min(len(a), len(b)); return a[:n]-b[:n]

def norm_of(root, tag):
    m = jload(os.path.join(ROOTS[root], "adapters", tag, "train_meta.json")) or {}
    return m.get("lora_B_norm"), m.get("loss_tail")

ARMS = {}
print(f"{'arm':4s} {'adapter':16s} {'theta':>12s} {'||lora_B||':>11s} {'tail':>9s}")
for arm, ks, ko in (("A","A","B"), ("B","B","A")):
    rows = []
    for s in (0,1,2):
        t = f"{arm}_raw_s{s}_r16"; th = theta_of(BASE, t, ks, ko)
        if not len(th): continue
        b, l = norm_of("base", t); rows.append((t, "base", th, b, l))
    if EXT is not None:
        for s in (3,4,5):
            t = f"{arm}_raw_s{s}_r16"; th = theta_of(EXT, t, ks, ko)
            if not len(th): continue
            b, l = norm_of("seedext", t); rows.append((t, "ext", th, b, l))
    means = np.array([r[2].mean() for r in rows]); k = len(means)
    per_img = np.concatenate([r[2] for r in rows])
    se_cl = float(means.std(ddof=1)/np.sqrt(k)); tc = float(sps.t.ppf(1-ALPHA_SIM/2, df=k-1))
    se_img = float(per_img.std(ddof=1)/np.sqrt(len(per_img)))
    ARMS[arm] = {"n_adapters": k, "adapters": [r[0] for r in rows],
                 "means": means.tolist(), "theta": float(means.mean()),
                 "sd": float(means.std(ddof=1)), "se_cluster": se_cl, "t_crit": tc,
                 "U_cluster": float(means.mean()+tc*se_cl),
                 "U_image": float(per_img.mean()+sps.norm.ppf(1-ALPHA_SIM/2)*se_img),
                 "norms": [r[3] for r in rows], "tails": [r[4] for r in rows]}
    for r in rows:
        print(f"{arm:4s} {r[0]:16s} {r[2].mean():+12.3e} "
              f"{(r[3] if r[3] is not None else float('nan')):11.2f} "
              f"{(r[4] if r[4] is not None else float('nan')):9.5f}")
    print(f"     -> n={k} theta={means.mean():+.3e} sd={means.std(ddof=1):.3e} "
          f"t={tc:.2f} U_cluster={ARMS[arm]['U_cluster']:+.3e} "
          f"U_image={ARMS[arm]['U_image']:+.3e}\n")

U = max(ARMS["A"]["U_cluster"], ARMS["B"]["U_cluster"])
theta_sym = 0.5*(ARMS["A"]["theta"]+ARMS["B"]["theta"])
NUM.update({"U_device": U, "theta_sym": theta_sym,
            "lambda_hat": theta_sym/NUM["R_real"],
            "lambda_U_plugin": U/NUM["R_real"],
            "lambda_U_conservative": U/NUM["R_real_L99"]})
if "R_vae_post" in NUM:
    NUM["tau_U_plugin"] = U/NUM["R_vae_post"]; NUM["tau_U_conservative"] = U/NUM["R_vae_L99"]
print("="*72)
print(f"U_device (max of arms)         = {U:+.4e}")
print(f"theta_sym                      = {theta_sym:+.4e}   lambda_hat = {NUM['lambda_hat']:+.4%}")
print(f"lambda_U  plug-in denominator  = {NUM['lambda_U_plugin']:.4%}")
print(f"lambda_U  conservative (L99)   = {NUM['lambda_U_conservative']:.4%}   <- report this")
if "tau_U_plugin" in NUM:
    print(f"tau_U     plug-in              = {NUM['tau_U_plugin']:.4%}")
    print(f"tau_U     conservative         = {NUM['tau_U_conservative']:.4%}   <- report this")
print("="*72)

In [ ]:
# %% C3 — leave-one-adapter-out and the strength covariate (exploratory)
print("leave-one-adapter-out")
for arm in ("A","B"):
    m = np.array(ARMS[arm]["means"]); ids = ARMS[arm]["adapters"]
    for i in range(len(m)):
        k_ = np.delete(m, i); n_ = len(k_)
        U_ = k_.mean()+sps.t.ppf(1-ALPHA_SIM/2, df=n_-1)*k_.std(ddof=1)/np.sqrt(n_)
        print(f"  {arm}  drop {ids[i]:16s} theta={k_.mean():+.3e} sd={k_.std(ddof=1):.3e} U={U_:+.3e}")
print("\ntheta vs adaptation strength  (EXPLORATORY — n is small, report both measures)")
for arm in ("A","B"):
    m = np.array(ARMS[arm]["means"], float)
    b = np.array([x if x is not None else np.nan for x in ARMS[arm]["norms"]], float)
    t = np.array([x if x is not None else np.nan for x in ARMS[arm]["tails"]], float)
    ok = ~np.isnan(b)
    if ok.sum() >= 3:
        pr, pp = sps.pearsonr(m[ok], b[ok]); sr, sp = sps.spearmanr(m[ok], b[ok])
        print(f"  arm {arm}: theta~||lora_B||  Pearson {pr:+.3f} (p={pp:.3f})  "
              f"Spearman {sr:+.3f} (p={sp:.3f})   n={ok.sum()}")
        ok2 = ~np.isnan(t)
        if ok2.sum() >= 3:
            pr2, pp2 = sps.pearsonr(m[ok2], t[ok2])
            print(f"          theta~tail_loss   Pearson {pr2:+.3f} (p={pp2:.3f})")
print("\nNOTE: a Pearson driven by one leverage point is not a relationship — check Spearman.")

In [ ]:
# %% C4 — E-AMP: loss-side sweep, generation-side sweep, specificity controls
AMP = cload(INV[("amp","csv/a4_measure.csv")])
amp_meta = []
for d in sorted(glob.glob(os.path.join(ROOTS["amp"], "adapters", "*"))):
    m = jload(os.path.join(d, "train_meta.json"))
    if m: amp_meta.append(m)
if amp_meta:
    base_tail = next((m["loss_tail"] for m in amp_meta
                      if m.get("kind")=="amp" and float(m.get("alpha",0))==1.0), None)
    print("LOSS SIDE (paired: all arms share seed 0)")
    print(f"{'tag':12s} {'kind':6s} {'alpha':>6s} {'tail':>9s} {'delta':>11s} {'||B||':>8s}")
    for m in sorted(amp_meta, key=lambda z: (z.get("kind",""), float(z.get("alpha",0)))):
        d_ = m["loss_tail"]-base_tail if base_tail else float("nan")
        print(f"{m['tag']:12s} {m.get('kind',''):6s} {float(m.get('alpha',0)):6.1f} "
              f"{m['loss_tail']:9.5f} {d_:+11.2e} {m.get('lora_B_norm',float('nan')):8.2f}")
    print("  wording: observed ONSET ~3x in ONE paired sweep — not a formal threshold.")

if AMP is not None:
    def v(tag, K):
        return AMP[(AMP.tag==tag)&(AMP.K==K)].sort_values("gen_idx")["rho_mult"].to_numpy(float)
    def paired(tag, tgt, ref="A"):
        a, b = v(tag, tgt), v(tag, ref); n = min(len(a), len(b)); return a[:n]-b[:n]
    print("\nGENERATION SIDE (paired contrast vs K_A)")
    print(f"{'tag':12s} {'rho(->K_B)':>12s} {'contrast':>12s} {'t':>7s}")
    for m in sorted([z for z in amp_meta if z.get("kind")=="amp"], key=lambda z: float(z["alpha"])):
        t_ = m["tag"]; ctr = paired(t_, "B")
        if not len(ctr): continue
        print(f"{t_:12s} {v(t_,'B').mean():12.3e} {ctr.mean():+12.3e} "
              f"{ctr.mean()/(ctr.std(ddof=1)/np.sqrt(len(ctr))):7.2f}")
    print("\nSPECIFICITY CONTROLS")
    for tag, tgt in (("ctrl_Q","Q"), ("ctrl_shift","Bshift")):
        if tag not in set(AMP.tag): continue
        for target, label in ((tgt, "correct template"), ("B", "mismatch (must be null)")):
            ctr = paired(tag, target)
            if not len(ctr): continue
            se = ctr.std(ddof=1)/np.sqrt(len(ctr))
            print(f"  {tag:11s} vs K_{target:7s} {ctr.mean():+.3e}  t={ctr.mean()/se:6.2f}  {label}")
    print("\n  ctrl_Q's comparator is K_A, whose generic offset need not match K_Q's — treat the")
    print("  magnitude of that negative as comparator-affected. K_Bshift shares K_B's spectrum.")

In [ ]:
# %% C5 — E-POST detector calibration IN GENERATED IMAGES, with intercept
LOW  = cload(INV[("amp","csv/epost_low.csv")])
LOWQ = cload(INV[("amp","csv/epost_lowq.csv")])
CAL = {}

def fit_cal(d, label, alpha_col="alpha", val="contrast"):
    rows = []
    for a in sorted(d[alpha_col].unique()):
        x = d[d[alpha_col]==a][val].to_numpy(float)
        m, se = float(x.mean()), float(x.std(ddof=1)/np.sqrt(len(x)))
        rows.append((float(a), len(x), m, se))
    A = np.array([r[0] for r in rows]); M = np.array([r[2] for r in rows])
    W = 1.0/np.array([r[3] for r in rows])**2
    X = np.vstack([np.ones_like(A), A]).T; Wm = np.diag(W)
    beta = np.linalg.solve(X.T@Wm@X, X.T@Wm@M); cov = np.linalg.inv(X.T@Wm@X)
    b0, b1 = float(beta[0]), float(beta[1]); se1 = float(np.sqrt(cov[1,1]))
    L_b1 = b1 - sps.norm.ppf(0.99)*se1
    det = [r[0] for r in rows if r[0]>0 and r[2]-sps.norm.ppf(0.995)*r[3] > 0]
    deg = np.allclose(M, M[0])
    print(f"\n── {label} ──")
    print(f"{'alpha':>7s} {'n':>6s} {'mean':>12s} {'SE':>11s} {'t':>7s} {'CI>0':>6s}")
    for a, n, m, se in rows:
        print(f"{a:7.3f} {n:6d} {m:+12.4e} {se:11.3e} {m/se if se else 0:7.2f} "
              f"{'YES' if m-sps.norm.ppf(0.995)*se>0 else '-':>6s}")
    if deg:
        print("  *** DEGENERATE: identical mean at every alpha. If this is a 'quantised' arm, the")
        print("  *** injection was a NO-OP: an 8-bit PNG is integer-valued, so re-rounding a")
        print("  *** sub-0.5-LSB perturbation returns the same array. Needs dither reconstruction.")
        return None
    a_eq = (NUM["U_device"]-b0)/L_b1
    print(f"  fit C = {b0:+.3e} + {b1:.4e}*alpha   lower99 slope {L_b1:.4e}")
    print(f"  alpha_equiv of U (conservative) = {100*a_eq:.4f}% of natural   margin {1/a_eq:.0f}x")
    print(f"  smallest alpha DIRECTLY detected = {min(det) if det else 'none'}"
          f"   -> extrapolation gap {min(det)/a_eq:.1f}x" if det else "")
    return {"b0":b0,"b1":b1,"se_b1":se1,"L_b1":L_b1,"alpha_equiv":a_eq,
            "smallest_detected":min(det) if det else None,"rows":rows}

if LOW is not None:
    for q, lab in ((0,"E-POST float (no requantisation)"), (1,"E-POST re-rounded")):
        sub = LOW[LOW["quant"]==q]
        if len(sub): CAL[lab] = fit_cal(sub, lab)
if LOWQ is not None and len(LOWQ):
    CAL["E-POST dither+quantise"] = fit_cal(LOWQ, "E-POST dither+quantise (full pipeline)")
best = CAL.get("E-POST dither+quantise") or CAL.get("E-POST float (no requantisation)")
if best:
    NUM["alpha_equiv"] = best["alpha_equiv"]; NUM["cal_slope_L99"] = best["L_b1"]
    print(f"\nHEADLINE SENSITIVITY: bound = {100*best['alpha_equiv']:.3f}% of natural amplitude "
          f"({1/best['alpha_equiv']:.0f}x margin)")

In [ ]:
# %% C6 — figures
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
OUT = os.path.join(ROOTS["base"], "figs_paper"); os.makedirs(OUT, exist_ok=True)

# F1 — the localisation: every adapter, both arms, the bound, and the two reference contrasts
fig, ax = plt.subplots(figsize=(7.2,4.2))
for i,(arm,col) in enumerate((("A","tab:blue"),("B","tab:orange"))):
    m = np.array(ARMS[arm]["means"]); x = np.full(len(m), i)+np.linspace(-.12,.12,len(m))
    ax.scatter(x, m, s=26, color=col, zorder=3, label=f"arm {arm} adapters (n={len(m)})")
    ax.hlines(ARMS[arm]["theta"], i-.2, i+.2, color=col, lw=2)
ax.axhline(0, color="k", lw=.6)
ax.axhline(NUM["U_device"], color="crimson", ls="--", lw=1.4,
           label=f"99% simultaneous upper limit = {NUM['U_device']:.2e}")
ax.set_xticks([0,1]); ax.set_xticklabels(["arm A","arm B"])
ax.set_ylabel(r"adapter-level device contrast $\theta$")
ax.set_title("Generation-side device transfer: every adapter, and the bound")
ax.legend(fontsize=7); fig.savefig(os.path.join(OUT,"F1_bound.png"), dpi=170, bbox_inches="tight")
plt.close(fig)

# F2 — stage localisation on one axis, log scale
fig, ax = plt.subplots(figsize=(6.4,4))
labels = ["real images\n(R_real)", "after VAE\n(R_VAE)", "generation\n(upper bound)"]
vals = [NUM["R_real"], NUM.get("R_vae_post", np.nan), NUM["U_device"]]
ax.bar(range(3), vals, color=["tab:green","tab:olive","crimson"])
for i,v_ in enumerate(vals):
    if np.isfinite(v_): ax.text(i, v_*1.15, f"{v_:.2e}", ha="center", fontsize=8)
ax.set_yscale("log"); ax.set_xticks(range(3)); ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel("device-specific contrast"); ax.set_title("Where device identity is lost")
fig.savefig(os.path.join(OUT,"F2_localisation.png"), dpi=170, bbox_inches="tight"); plt.close(fig)

# F3 — detector calibration, real vs generated, with the bound drawn on
REAL = cload(INV[("amp","csv/a1_real_dose.csv")])
fig, ax = plt.subplots(figsize=(6.6,4.2))
if REAL is not None:
    r = REAL.groupby("alpha")["rho_KB"].median()
    ax.plot(r.index, r.values, "o-", label="real images (reference)")
for lab, cal in CAL.items():
    if cal is None: continue
    A_ = [x[0] for x in cal["rows"]]; M_ = [x[2] for x in cal["rows"]]
    ax.plot(A_, M_, "s-", label=f"generated — {lab}")
ax.axhline(NUM["U_device"], color="crimson", ls="--", lw=1.2, label="upper bound on transfer")
ax.set_xscale("symlog", linthresh=1e-2); ax.set_yscale("symlog", linthresh=1e-5)
ax.set_xlabel(r"injected amplitude $\alpha$ (× natural)"); ax.set_ylabel("measured contrast")
ax.set_title("Detector calibration"); ax.legend(fontsize=7)
fig.savefig(os.path.join(OUT,"F3_calibration.png"), dpi=170, bbox_inches="tight"); plt.close(fig)

# F4 — the paired design working: generic offset vs the contrast that survives it
if AMP is not None and len(amp_meta):
    fig, ax = plt.subplots(figsize=(6.6,4))
    tags = [m["tag"] for m in sorted([z for z in amp_meta if z.get("kind")=="amp"],
                                     key=lambda z: float(z["alpha"]))]
    off = [v(t_, "B").mean() for t_ in tags]; ctr = [paired(t_, "B").mean() for t_ in tags]
    x = np.arange(len(tags))
    ax.bar(x-.2, off, .4, label=r"raw $\rho(\to K_B)$  (generic offset)")
    ax.bar(x+.2, ctr, .4, label="paired contrast (offset removed)")
    ax.axhline(0, color="k", lw=.6); ax.set_xticks(x)
    ax.set_xticklabels([t_.replace("amp_","α=").replace("p",".") for t_ in tags], fontsize=8)
    ax.set_ylabel("mean correlation"); ax.set_title("The generic offset, and what the pairing removes")
    ax.legend(fontsize=7)
    fig.savefig(os.path.join(OUT,"F4_pairing.png"), dpi=170, bbox_inches="tight"); plt.close(fig)
print("figures ->", OUT)
print(sorted(os.listdir(OUT)))

In [ ]:
# %% C7 — consistency audit against cached results, and what is still missing
print("── recomputed vs cached ──")
cached = {**(jload(INV[("base","E_INV_P0_results.json")]) or {}),
          "seedext": jload(INV[("seedext","seedext_pooled.json")]) or {}}
def cmp(name, mine, theirs, tol=0.02):
    if theirs is None: print(f"  {name:28s} {mine:+.4e}   (no cached value)"); return
    rel = abs(mine-theirs)/max(abs(theirs),1e-30)
    print(f"  {name:28s} {mine:+.4e} vs {theirs:+.4e}   "
          f"{'OK' if rel<tol else '*** DISAGREE ***'} ({100*rel:.2f}%)")
cmp("R_real", NUM["R_real"], None)
cmp("U_device", NUM["U_device"], cached.get("seedext",{}).get("U_device"))
cmp("lambda_U (plug-in)", NUM["lambda_U_plugin"], cached.get("seedext",{}).get("lambda_U"))

print("\n── still missing ──")
todo = []
if INV[("amp","csv/epost_lowq.csv")] is None or (LOWQ is not None and not len(LOWQ)):
    todo.append("E-POST-LOW-Q: dither-reconstruction quantisation arm (GPU, ~47 min)")
if INV[("shift","csv/r3_measure.csv")] is None:
    todo.append("E-SHIFT-REP: 5 new seeds per arm, seed 0 = discovery, confirm on new seeds only")
todo += ["hierarchical bootstrap on the ratio (sensitivity alongside the conservative limits)",
         "E5 second fingerprint extractor",
         "E-VAE-SCREEN -> E4 second generative system",
         "E-MULTIDEV crossed 5 devices x 2 seeds",
         "prior-art sweep: Scholar forward-cites on Chen-Fridrich-Goljan 2008 and SIREN; "
         "arXiv full-text PRNU x LoRA; WIFS 2026 proceedings"]
for t_ in todo: print("  •", t_)

json.dump({k:(float(v) if isinstance(v,(int,float,np.floating)) else v)
           for k,v in NUM.items()},
          open(os.path.join(ROOTS["base"],"CONSOLIDATED_NUMBERS.json"),"w"), indent=2)
print("\nnumbers -> CONSOLIDATED_NUMBERS.json")
print("\n── paper-ready sentence ──")
print(f"Device identity survives latent encoding, retaining {100*NUM.get('eta_device',float('nan')):.1f}% "
      f"of the same-model device contrast. After adaptation and text-to-image generation,\n"
      f"device-specific transfer remains below a simultaneous adapter-clustered upper bound of "
      f"{NUM['lambda_U_conservative']:.3%} of the real-image contrast"
      + (f", equivalently {NUM['tau_U_conservative']:.3%} of the signal transmitted by the encoder."
         if "tau_U_conservative" in NUM else "."))

## Notes

- **Everything is recomputed from per-row CSVs**, so C7's audit catches drift between a cached
  summary and what the data actually says. A `*** DISAGREE ***` line is worth chasing.
- **Conservative limits are the ones to report.** λ_U uses the *lower* 99% limit of R_real, and
  α_equiv uses the *lower* 99% limit of the calibration slope with the intercept subtracted. The
  plug-in values are shown alongside for comparison only.
- **C5 detects the degenerate quantisation arm automatically.** If a "quantised" calibration returns
  an identical mean at every α, the injection was a no-op — an 8-bit PNG is integer-valued, so
  re-rounding a sub-0.5-LSB perturbation returns the same array. The fix is dither reconstruction,
  not more images.
- **Nothing here needs a GPU.** Re-running the missing GPU experiments is separate; this notebook
  only tells you what you have and what each number really is.
- Runs off a local copy too — set `EINV_DRIVE` to a directory containing `inv_channel/`.